# E-Commerce Competitive Intelligence Multi-Agent System
## Week 8 Assignment: Production Deployment & UI

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Learning Objectives

- Create Gradio interactive interface
- Deploy as a FastAPI application
- Complete production readiness steps

## Difficulty: Advanced

**IMPORTANT**: Run all prior week cells first before starting this week's exercises.

---

---
# WEEK 1: Environment Setup & Langfuse Initialization (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 1: Environment Setup & Langfuse Initialization

---

## Learning Objectives

By the end of Week 1, you will be able to:

- [ ] Configure a production-ready development environment
- [ ] Initialize Langfuse observability from the first line of code
- [ ] Understand e-commerce product data structures
- [ ] Perform exploratory data analysis on product catalogs
- [ ] Create traced wrapper functions for API calls

## 1.1 Package Installation

We install packages in a specific order to avoid dependency conflicts. Each package serves a distinct purpose in our competitive intelligence architecture.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## Key Concept: Package Architecture for E-Commerce AI

Understanding why we use each package helps you make informed architectural decisions:

| Package | Purpose | E-Commerce Use Case |
|---------|---------|---------------------|
| `langchain` | Orchestration framework | Coordinate multiple analysis agents |
| `langfuse` | Observability & tracing | Track every pricing decision, audit trail |
| `chromadb` | Vector database | Semantic product search & matching |
| `networkx` | Graph algorithms | Product relationship mapping |
| `pydantic` | Data validation | Structured agent outputs |
| `gradio` | UI framework | Business user interface |

### Why Observability First?

In competitive intelligence, every AI decision can impact business outcomes:
- A pricing recommendation could affect millions in revenue
- Marketing claims must be factually defensible
- Feature comparisons need accuracy for legal compliance

**Langfuse provides the audit trail** that makes AI decisions transparent and reviewable.

## 1.2 Environment Detection & Configuration

This section handles detecting our runtime environment (Colab vs local) and loading API keys securely.

In [ ]:
# ============================================================================
# WEEK 1.2: CORE IMPORTS
# ============================================================================

import os
import sys
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

print("Core imports completed successfully")

In [ ]:
# ============================================================================
# ENVIRONMENT DETECTION
# ============================================================================

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

print(f"Environment Detection:")
print(f"  Running in Colab: {IN_COLAB}")
print(f"  Python version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION & DATA INGESTION
# ============================================================================
# API keys from Colab Secrets or .env file
# Datasets fetched programmatically from GitHub

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "ecommerce"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)


# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

In [ ]:
# ============================================================================
# PROJECT CONFIGURATION
# ============================================================================

PROJECT_NAME = "ecommerce-competitive-intelligence"
DATA_DIR = Path(DATA_DIR_BASE)

print("\nProject Configuration:")
print(f"  Project Name: {PROJECT_NAME}")
print(f"  Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"  Data Directory: {DATA_DIR}")
print(f"  Timestamp: {datetime.now().isoformat()}")

## Deep Dive: Secure API Key Management in E-Commerce

E-commerce competitive intelligence systems often process sensitive data. Proper API key management is critical:

### Security Best Practices

| Environment | Method | Security Level |
|-------------|--------|----------------|
| Development | `.env` file (gitignored) | Medium |
| Colab | Colab Secrets | High |
| Production | Secret Manager (AWS/GCP) | Highest |

### What NOT to Do

```python
# NEVER do this - keys will leak to version control!
OPENAI_API_KEY = "sk-abc123..."  # BAD!
```

### Why This Matters for E-Commerce

- **Competitive data is sensitive**: Your pricing strategies shouldn't leak
- **API costs can spiral**: Exposed keys get abused quickly
- **Compliance requirements**: Many industries require audit trails for AI decisions

## 1.3 Langfuse Initialization

Now we initialize Langfuse - the **foundation of our observability strategy**. Every subsequent operation will be traceable.

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================

from langfuse import Langfuse
try:
    from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
except (ImportError, ModuleNotFoundError):
    try:
        from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler
    except (ImportError, ModuleNotFoundError):
        # Langfuse v3+: CallbackHandler not needed, create a no-op
        class LangfuseCallbackHandler:
            def __init__(self, **kwargs): pass
from openai import OpenAI

# Initialize Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
    public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')
)

# Verify connection
try:
    langfuse.auth_check()
    print("Langfuse connection verified!")
except Exception as e:
    print(f"Langfuse connection failed: {e}")
    raise

In [ ]:
# ============================================================================
# SESSION CONFIGURATION
# ============================================================================

# Create a unique session ID for this notebook run
# This groups all traces from this session together
SESSION_ID = f"ecommerce-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"\nSession Configuration:")
print(f"  Session ID: {SESSION_ID}")
print(f"  Langfuse Host: {os.environ.get('LANGFUSE_HOST')}")
print(f"  Dashboard URL: {os.environ.get('LANGFUSE_HOST')}/project")

## Key Concept: Session IDs and Trace Organization

In Langfuse, traces are organized hierarchically for easy navigation:

```
Project: ecommerce-competitive-intelligence
  |
  +-- Session: ecommerce-20240115-143022
       |
       +-- Trace: load-product-catalogs
       |    +-- Span: parse-company-x
       |    +-- Span: parse-company-y
       |
       +-- Trace: price-analysis-headphones
       |    +-- Generation: gpt-4o-mini
       |
       +-- Trace: orchestration-full-analysis
            +-- Span: parallel-agents
            +-- Span: aggregation
```

### Benefits for E-Commerce Intelligence

1. **Debugging**: Find exactly which agent made which recommendation
2. **Cost tracking**: See API costs per analysis type
3. **Performance**: Identify bottlenecks in multi-agent workflows
4. **Audit**: Full trail for compliance and review

## 1.4 Traced Wrapper Functions

We create wrapper functions that automatically trace all OpenAI API calls. This ensures **100% observability** without cluttering application code.

In [ ]:
# ============================================================================
# TRACED EMBEDDING FUNCTION
# ============================================================================

def traced_embedding(
    text: str,
    trace_name: str = "embedding",
    metadata: Dict = None
) -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Every embedding operation is logged with:
    - Input text (preview)
    - Model used
    - Token usage
    - Output dimensions

    Args:
        text: Text to embed
        trace_name: Name for the trace
        metadata: Additional metadata to log

    Returns:
        List of embedding values
    """
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "text_length": len(text),
            "text_preview": text[:100],
            **(metadata or {})
        }
    )

    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text[:500]  # Log preview of input
    )

    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    generation.end(
        output={"dimensions": len(embedding)},
        usage={"total_tokens": response.usage.total_tokens}
    )

    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(
    prompt: str,
    system: str = "",
    trace_name: str = "completion",
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    metadata: Dict = None
) -> str:
    """
    Generate completion with full Langfuse tracing.

    Logs:
    - Full prompt and system message
    - Model configuration
    - Token usage (input/output/total)
    - Complete response

    Args:
        prompt: User prompt
        system: System message
        trace_name: Name for the trace
        model: Model to use
        temperature: Sampling temperature
        metadata: Additional metadata

    Returns:
        Model response text
    """
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "model": model,
            "temperature": temperature,
            **(metadata or {})
        }
    )

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    generation = trace.generation(
        name="openai-completion",
        model=model,
        input=messages
    )

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )

    result = response.choices[0].message.content

    generation.end(
        output=result,
        usage={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    )

    return result

print("traced_completion() function defined")

In [ ]:
# ============================================================================
# LANGCHAIN CALLBACK HANDLER
# ============================================================================

def get_langfuse_handler(
    trace_name: str,
    tags: List[str] = None
) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    Use this when working with LangChain chains and agents
    to get automatic tracing of all LLM calls.

    Args:
        trace_name: Name for the trace
        tags: List of tags for filtering

    Returns:
        Configured LangfuseCallbackHandler
    """
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or []
    )

print("get_langfuse_handler() function defined")
print("\nTraced wrapper functions ready:")
print("  - traced_embedding()")
print("  - traced_completion()")
print("  - get_langfuse_handler()")

## Example 1: Testing Traced Functions

Let's verify our traced functions work correctly. Check Langfuse dashboard to see the traces appear.

In [ ]:
# ============================================================================
# EXAMPLE 1: Test Embedding Function
# ============================================================================

print("EXAMPLE 1: Testing traced_embedding()")
print("="*50)

test_product = "Wireless Bluetooth headphones with active noise cancellation, 30-hour battery life, and premium sound quality."

embedding = traced_embedding(
    text=test_product,
    trace_name="test-product-embedding",
    metadata={"product_type": "headphones"}
)

print(f"Input: {test_product[:50]}...")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print("\nCheck Langfuse dashboard for trace: test-product-embedding")

In [ ]:
# ============================================================================
# EXAMPLE 2: Test Completion Function
# ============================================================================

print("\nEXAMPLE 2: Testing traced_completion()")
print("="*50)

response = traced_completion(
    prompt="Compare these two products and identify which offers better value:\n\nProduct A: Wireless headphones, $99, 20hr battery, noise cancelling\nProduct B: Wireless headphones, $129, 30hr battery, noise cancelling, waterproof",
    system="You are a product comparison expert. Be concise.",
    trace_name="test-product-comparison",
    metadata={"comparison_type": "value"}
)

print(f"Response:\n{response}")
print("\nCheck Langfuse dashboard for trace: test-product-comparison")

## 1.5 E-Commerce Product Catalogs

Now we load our product data. In a real system, this would come from APIs or databases. Here we use sample catalogs that represent typical e-commerce data.

In [ ]:
# ============================================================================
# WEEK 1.5: PRODUCT CATALOG - COMPANY X (Our Company)
# ============================================================================

# Trace data loading
data_trace = langfuse.trace(
    name="load-product-catalogs",
    session_id=SESSION_ID,
    tags=["data-loading", "week-1"]
)

# Company X (Our Company) Product Catalog
COMPANY_X_CATALOG = {
    "company": "Company X",
    "description": "Premium consumer electronics brand",
    "products": [
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones X1",
            "price": 99.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Noise Cancelling", "20h Battery", "Foldable"],
            "discount": "10% off",
            "availability": "In Stock",
            "sku": "X1-HP-001"
        },
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones X2 Pro",
            "price": 149.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Advanced ANC", "30h Battery", "Foldable", "USB-C", "Multipoint"],
            "discount": None,
            "availability": "In Stock",
            "sku": "X2-HP-002"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch X1",
            "price": 199.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "5 Day Battery", "Water Resistant", "Sleep Tracking"],
            "discount": "15% off",
            "availability": "In Stock",
            "sku": "X1-SW-001"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch X2 Ultra",
            "price": 349.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "ECG", "10 Day Battery", "Titanium", "Dive Mode"],
            "discount": None,
            "availability": "Limited Stock",
            "sku": "X2-SW-002"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker X1",
            "price": 79.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Waterproof", "12h Battery", "360 Sound"],
            "discount": None,
            "availability": "In Stock",
            "sku": "X1-SP-001"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker X2 Party",
            "price": 149.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Waterproof", "24h Battery", "LED Lights", "Mic Input"],
            "discount": "20% off",
            "availability": "In Stock",
            "sku": "X2-SP-002"
        }
    ]
}

print(f"Company X Catalog Loaded:")
print(f"  Company: {COMPANY_X_CATALOG['company']}")
print(f"  Products: {len(COMPANY_X_CATALOG['products'])}")

In [ ]:
# ============================================================================
# PRODUCT CATALOG - COMPANY Y (Competitor)
# ============================================================================

# Company Y (Competitor) Product Catalog
COMPANY_Y_CATALOG = {
    "company": "Company Y",
    "description": "Value-focused consumer electronics",
    "products": [
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones Z1",
            "price": 105.00,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Noise Cancelling", "25h Battery", "Quick Charge", "Foldable"],
            "discount": "5% off + Free Case",
            "availability": "In Stock",
            "sku": "Z1-HP-001"
        },
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones Z2",
            "price": 115.00,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Advanced Noise Cancelling", "30h Battery", "Waterproof"],
            "discount": None,
            "availability": "In Stock",
            "sku": "Z2-HP-002"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch Z1",
            "price": 189.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "7 Day Battery", "Water Resistant", "Sleep Tracking", "SpO2"],
            "discount": "20% off",
            "availability": "In Stock",
            "sku": "Z1-SW-001"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch Z2 Sport",
            "price": 279.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "5 Day Battery", "100+ Workouts", "Voice Assistant"],
            "discount": "10% off",
            "availability": "In Stock",
            "sku": "Z2-SW-002"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker Z1",
            "price": 69.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Waterproof", "15h Battery", "Stereo Pairing"],
            "discount": "Free Shipping",
            "availability": "In Stock",
            "sku": "Z1-SP-001"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker Z2 Boom",
            "price": 129.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "IP67 Rating", "20h Battery", "Bass Boost", "Daisy Chain"],
            "discount": None,
            "availability": "In Stock",
            "sku": "Z2-SP-002"
        }
    ]
}

print(f"Company Y Catalog Loaded:")
print(f"  Company: {COMPANY_Y_CATALOG['company']}")
print(f"  Products: {len(COMPANY_Y_CATALOG['products'])}")

In [ ]:
# ============================================================================
# LOG CATALOG SUMMARY TO LANGFUSE
# ============================================================================

data_trace.update(output={
    "company_x_products": len(COMPANY_X_CATALOG['products']),
    "company_y_products": len(COMPANY_Y_CATALOG['products']),
    "total_products": len(COMPANY_X_CATALOG['products']) + len(COMPANY_Y_CATALOG['products']),
    "categories": list(set(
        [p['category'] for p in COMPANY_X_CATALOG['products']] +
        [p['category'] for p in COMPANY_Y_CATALOG['products']]
    ))
})

print("\nCatalog Summary:")
print(f"  Total Products: {len(COMPANY_X_CATALOG['products']) + len(COMPANY_Y_CATALOG['products'])}")
print(f"  Company X: {len(COMPANY_X_CATALOG['products'])} products")
print(f"  Company Y: {len(COMPANY_Y_CATALOG['products'])} products")

## Key Concept: E-Commerce Product Data Structure

Understanding product data structure is essential for competitive analysis:

### Core Product Attributes

| Attribute | Purpose | Analysis Use |
|-----------|---------|---------------|
| `category` | Product classification | Category-level comparison |
| `product_name` | Unique identifier | Matching & display |
| `price` | Base price | Price positioning |
| `features` | Product capabilities | Feature gap analysis |
| `discount` | Current promotions | Effective price calculation |
| `availability` | Stock status | Opportunity detection |

### Why Features Are Lists

Features as lists enable:
- Set operations (intersection, difference)
- Feature counting and scoring
- Semantic similarity matching
- Gap analysis

## 1.6 Exploratory Data Analysis (EDA)

Before building our intelligence system, we need to understand the data. EDA reveals patterns and informs our agent design.

In [ ]:
# ============================================================================
# WEEK 1.6: BUILD EDA DATAFRAME
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Trace EDA
eda_trace = langfuse.trace(
    name="exploratory-data-analysis",
    session_id=SESSION_ID,
    tags=["eda", "visualization", "week-1"]
)

# Build comprehensive EDA dataframe
eda_data = []

for product in COMPANY_X_CATALOG['products']:
    # Parse discount percentage
    discount_pct = 0
    if product['discount']:
        match = re.search(r'(\d+)%', str(product['discount']))
        if match:
            discount_pct = int(match.group(1))

    eda_data.append({
        'company': 'Company X',
        'category': product['category'],
        'product': product['product_name'],
        'base_price': product['price'],
        'discount_pct': discount_pct,
        'effective_price': round(product['price'] * (1 - discount_pct/100), 2),
        'feature_count': len(product['features']),
        'has_discount': product['discount'] is not None,
        'in_stock': product['availability'] == 'In Stock'
    })

for product in COMPANY_Y_CATALOG['products']:
    discount_pct = 0
    if product['discount']:
        match = re.search(r'(\d+)%', str(product['discount']))
        if match:
            discount_pct = int(match.group(1))

    eda_data.append({
        'company': 'Company Y',
        'category': product['category'],
        'product': product['product_name'],
        'base_price': product['price'],
        'discount_pct': discount_pct,
        'effective_price': round(product['price'] * (1 - discount_pct/100), 2),
        'feature_count': len(product['features']),
        'has_discount': product['discount'] is not None,
        'in_stock': product['availability'] == 'In Stock'
    })

eda_df = pd.DataFrame(eda_data)

print("EDA DataFrame created")
print(f"Shape: {eda_df.shape}")
print(f"\nColumns: {list(eda_df.columns)}")

In [ ]:
# ============================================================================
# EDA SUMMARY STATISTICS
# ============================================================================

print("PRODUCT CATALOG SUMMARY")
print("="*60)

print(f"\nTotal products: {len(eda_df)}")

print(f"\nProducts by Company:")
print(eda_df['company'].value_counts().to_string())

print(f"\nProducts by Category:")
print(eda_df['category'].value_counts().to_string())

print(f"\nPrice Statistics by Company:")
print(eda_df.groupby('company')['effective_price'].agg(['mean', 'min', 'max']).round(2).to_string())

print(f"\nDiscount Statistics:")
print(f"  Products with discounts: {eda_df['has_discount'].sum()} / {len(eda_df)}")
print(f"  Average discount: {eda_df[eda_df['discount_pct'] > 0]['discount_pct'].mean():.1f}%")

In [ ]:
# ============================================================================
# EDA VISUALIZATION - Price Comparison
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Color palette
company_colors = {'Company X': '#3498db', 'Company Y': '#e74c3c'}

# 1. Effective Price by Product
ax1 = axes[0, 0]
x_products = eda_df[eda_df['company'] == 'Company X']
y_products = eda_df[eda_df['company'] == 'Company Y']

x = np.arange(len(x_products))
width = 0.35

ax1.bar(x - width/2, x_products['effective_price'], width, label='Company X', color='#3498db', alpha=0.8)
ax1.bar(x + width/2, y_products['effective_price'], width, label='Company Y', color='#e74c3c', alpha=0.8)
ax1.set_xlabel('Product Index')
ax1.set_ylabel('Effective Price ($)')
ax1.set_title('Effective Price Comparison')
ax1.legend()
ax1.set_xticks(x)

# 2. Average Price by Category
ax2 = axes[0, 1]
price_by_cat = eda_df.groupby(['category', 'company'])['effective_price'].mean().unstack()
price_by_cat.plot(kind='bar', ax=ax2, color=['#3498db', '#e74c3c'], alpha=0.8)
ax2.set_xlabel('Category')
ax2.set_ylabel('Average Effective Price ($)')
ax2.set_title('Average Price by Category')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Company')

# 3. Feature Count by Company
ax3 = axes[1, 0]
feature_by_company = eda_df.groupby('company')['feature_count'].mean()
colors = [company_colors[c] for c in feature_by_company.index]
feature_by_company.plot(kind='bar', ax=ax3, color=colors, alpha=0.8)
ax3.set_xlabel('Company')
ax3.set_ylabel('Average Feature Count')
ax3.set_title('Average Features per Product')
ax3.tick_params(axis='x', rotation=0)

# 4. Price vs Feature Count
ax4 = axes[1, 1]
for company in ['Company X', 'Company Y']:
    data = eda_df[eda_df['company'] == company]
    ax4.scatter(data['feature_count'], data['effective_price'],
                label=company, color=company_colors[company], s=100, alpha=0.7)
ax4.set_xlabel('Feature Count')
ax4.set_ylabel('Effective Price ($)')
ax4.set_title('Price vs Features')
ax4.legend()

plt.tight_layout()
plt.savefig('ecommerce_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEDA visualization saved to ecommerce_eda.png")

In [ ]:
# ============================================================================
# LOG EDA INSIGHTS TO LANGFUSE
# ============================================================================

eda_insights = {
    "total_products": len(eda_df),
    "avg_price_x": round(eda_df[eda_df['company'] == 'Company X']['effective_price'].mean(), 2),
    "avg_price_y": round(eda_df[eda_df['company'] == 'Company Y']['effective_price'].mean(), 2),
    "avg_features_x": round(eda_df[eda_df['company'] == 'Company X']['feature_count'].mean(), 2),
    "avg_features_y": round(eda_df[eda_df['company'] == 'Company Y']['feature_count'].mean(), 2),
    "discount_rate": round(eda_df['has_discount'].mean() * 100, 1)
}

eda_trace.update(output=eda_insights)

print("\nKey EDA Insights (logged to Langfuse):")
for key, value in eda_insights.items():
    print(f"  {key}: {value}")

## Checkpoint: Week 1 Verification

Before proceeding to Week 2, verify you have completed the following:

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Langfuse initialized", langfuse is not None),
    ("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 0),
    ("OpenAI client ready", openai_client is not None),
    ("traced_embedding() defined", 'traced_embedding' in globals()),
    ("traced_completion() defined", 'traced_completion' in globals()),
    ("Company X catalog loaded", len(COMPANY_X_CATALOG['products']) > 0),
    ("Company Y catalog loaded", len(COMPANY_Y_CATALOG['products']) > 0),
    ("EDA DataFrame created", len(eda_df) > 0),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 1 Complete! You may proceed to Week 2.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 1 Summary

### What We Accomplished

| Component | Status | Langfuse Traces |
|-----------|--------|------------------|
| Environment Setup | Complete | N/A |
| Langfuse Initialization | Complete | Session created |
| Traced Wrappers | Complete | test-* traces |
| Product Catalogs | Complete | load-product-catalogs |
| EDA | Complete | exploratory-data-analysis |

### Key Insights from EDA

1. **Pricing**: Company Y generally offers lower prices
2. **Features**: Both companies offer similar feature counts
3. **Discounts**: Both use promotional pricing strategies
4. **Opportunity**: Price-feature ratio analysis needed

### Next Steps (Week 2)

- Normalize product catalogs for comparison
- Extract comparable features
- Build product comparison framework

---
# WEEK 2: Data Processing & Feature Extraction (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 2: Data Processing & Feature Extraction

---

## Learning Objectives

By the end of Week 2, you will be able to:

- [ ] Normalize product catalogs with full tracing
- [ ] Extract and compare product features
- [ ] Calculate effective prices with discounts
- [ ] Build a product comparison framework
- [ ] Understand pricing strategies in e-commerce

## Deep Dive: E-Commerce Pricing Strategies

Before processing our data, let's understand the pricing strategies we'll analyze:

### Common Pricing Positions

| Position | Price vs Market | Feature Level | Target Customer |
|----------|-----------------|---------------|------------------|
| **Premium** | +20% or more | High | Quality-focused |
| **Competitive** | +/- 10% | Medium-High | Value-conscious |
| **Value** | -10% or more | Medium | Price-sensitive |
| **Budget** | -30% or more | Basic | Bargain hunters |

### Discount Types We'll Analyze

1. **Percentage Off**: "10% off" - Direct price reduction
2. **Bundle Deals**: "Free Case" - Added value
3. **Shipping**: "Free Shipping" - Hidden cost reduction
4. **Conditional**: "20% off orders over $100" - Threshold-based

### Why This Matters for AI Agents

Our Price Monitor Agent needs to:
- Calculate **effective prices** (after discounts)
- Understand **value perception** (features per dollar)
- Identify **pricing opportunities** (gaps in market)

## 2.1 Product Catalog Processor

We build a processor class that normalizes product data for comparison. Every operation is traced to Langfuse.

In [ ]:
# ============================================================================
# WEEK 2.1: PRODUCT CATALOG PROCESSOR CLASS
# ============================================================================

class TracedProductCatalogProcessor:
    """
    Process and normalize product catalogs with full Langfuse tracing.

    This class handles:
    - Discount parsing and effective price calculation
    - Feature normalization
    - Product ID generation
    - Comparison data preparation
    """

    def __init__(self):
        self.processed_products = []
        self.feature_vocabulary = set()

    def parse_discount(self, discount_str: str) -> Tuple[int, str]:
        """
        Parse discount string to extract percentage and type.

        Examples:
            "10% off" -> (10, "percentage")
            "Free Shipping" -> (0, "shipping")
            "5% off + Free Case" -> (5, "bundle")
        """
        if not discount_str:
            return 0, "none"

        discount_str = str(discount_str).lower()

        # Extract percentage
        match = re.search(r'(\d+)%', discount_str)
        pct = int(match.group(1)) if match else 0

        # Determine type
        if 'free' in discount_str and pct > 0:
            discount_type = "bundle"
        elif 'shipping' in discount_str:
            discount_type = "shipping"
        elif pct > 0:
            discount_type = "percentage"
        else:
            discount_type = "other"

        return pct, discount_type

    def normalize_features(self, features: List[str]) -> List[str]:
        """
        Normalize feature strings for comparison.

        - Converts to lowercase
        - Standardizes common terms
        - Adds to vocabulary
        """
        normalized = []
        for feature in features:
            norm = feature.lower().strip()
            # Standardize common variations
            norm = norm.replace('anc', 'noise cancelling')
            norm = norm.replace('nc', 'noise cancelling')
            normalized.append(norm)
            self.feature_vocabulary.add(norm)
        return normalized

    def normalize_product(self, product: Dict, company: str) -> Dict:
        """
        Normalize a single product record for comparison.
        """
        base_price = product.get('price', 0)
        discount_pct, discount_type = self.parse_discount(product.get('discount'))
        effective_price = base_price * (1 - discount_pct / 100)

        features = product.get('features', [])
        normalized_features = self.normalize_features(features)

        return {
            'company': company,
            'category': product.get('category', 'Unknown'),
            'product_name': product.get('product_name', 'Unknown'),
            'sku': product.get('sku', ''),
            'base_price': base_price,
            'discount_pct': discount_pct,
            'discount_type': discount_type,
            'effective_price': round(effective_price, 2),
            'currency': product.get('currency', 'USD'),
            'features': features,
            'features_normalized': normalized_features,
            'feature_count': len(features),
            'availability': product.get('availability', 'Unknown'),
            'discount_text': product.get('discount', ''),
            'price_per_feature': round(effective_price / max(len(features), 1), 2)
        }

print("TracedProductCatalogProcessor class defined")
print("  - parse_discount()")
print("  - normalize_features()")
print("  - normalize_product()")

In [ ]:
# ============================================================================
# CATALOG PROCESSING METHODS (continued)
# ============================================================================

# Add methods to the processor class
class TracedProductCatalogProcessor(TracedProductCatalogProcessor):

    def process_catalog_with_tracing(self, catalog: Dict) -> List[Dict]:
        """
        Process entire catalog with Langfuse tracing.

        Creates a trace for the catalog and spans for each product.
        """
        company = catalog.get('company', 'Unknown')
        products = catalog.get('products', [])

        trace = langfuse.trace(
            name=f"process-catalog-{company.lower().replace(' ', '-')}",
            session_id=SESSION_ID,
            metadata={"company": company, "product_count": len(products)},
            tags=["catalog-processing", "week-2"]
        )

        processed = []
        for i, product in enumerate(products):
            span = trace.span(
                name=f"normalize-{product.get('product_name', f'product-{i}')}"
            )

            normalized = self.normalize_product(product, company)
            normalized['product_id'] = f"{company.replace(' ', '')}_{len(processed)+1:03d}"
            processed.append(normalized)

            span.end(output={
                "product_id": normalized['product_id'],
                "effective_price": normalized['effective_price'],
                "feature_count": normalized['feature_count']
            })

        trace.update(output={
            "processed_count": len(processed),
            "avg_price": round(sum(p['effective_price'] for p in processed) / len(processed), 2),
            "avg_features": round(sum(p['feature_count'] for p in processed) / len(processed), 2)
        })

        self.processed_products.extend(processed)
        return processed

    def compare_products(self, product_x: Dict, product_y: Dict) -> Dict:
        """
        Compare two products head-to-head.

        Returns detailed comparison including:
        - Price advantage
        - Feature advantage
        - Unique features for each
        - Value score
        """
        price_diff = product_x['effective_price'] - product_y['effective_price']
        price_advantage = 'X' if price_diff < 0 else ('Y' if price_diff > 0 else 'tie')

        feature_diff = product_x['feature_count'] - product_y['feature_count']
        feature_advantage = 'X' if feature_diff > 0 else ('Y' if feature_diff < 0 else 'tie')

        x_features = set(f.lower() for f in product_x['features'])
        y_features = set(f.lower() for f in product_y['features'])

        # Calculate value score (features per dollar)
        value_x = product_x['feature_count'] / max(product_x['effective_price'], 1) * 100
        value_y = product_y['feature_count'] / max(product_y['effective_price'], 1) * 100
        value_advantage = 'X' if value_x > value_y else ('Y' if value_y > value_x else 'tie')

        return {
            'category': product_x['category'],
            'product_x': product_x['product_name'],
            'product_y': product_y['product_name'],
            'price_x': product_x['effective_price'],
            'price_y': product_y['effective_price'],
            'price_diff': round(abs(price_diff), 2),
            'price_diff_pct': round(abs(price_diff) / max(product_y['effective_price'], 1) * 100, 1),
            'price_advantage': price_advantage,
            'features_x': product_x['feature_count'],
            'features_y': product_y['feature_count'],
            'feature_advantage': feature_advantage,
            'value_x': round(value_x, 2),
            'value_y': round(value_y, 2),
            'value_advantage': value_advantage,
            'unique_to_x': list(x_features - y_features),
            'unique_to_y': list(y_features - x_features),
            'common_features': list(x_features & y_features)
        }

print("Additional methods added to TracedProductCatalogProcessor")
print("  - process_catalog_with_tracing()")
print("  - compare_products()")

In [ ]:
# ============================================================================
# PROCESS BOTH CATALOGS
# ============================================================================

# Initialize processor
processor = TracedProductCatalogProcessor()

# Process Company X catalog
print("Processing Company X catalog...")
products_x = processor.process_catalog_with_tracing(COMPANY_X_CATALOG)

# Process Company Y catalog
print("Processing Company Y catalog...")
products_y = processor.process_catalog_with_tracing(COMPANY_Y_CATALOG)

# Combine all products
all_products = products_x + products_y

print(f"\nProcessing Complete:")
print(f"  Company X: {len(products_x)} products")
print(f"  Company Y: {len(products_y)} products")
print(f"  Total: {len(all_products)} products")
print(f"  Feature Vocabulary: {len(processor.feature_vocabulary)} unique features")

In [ ]:
# ============================================================================
# DISPLAY PROCESSED PRODUCTS
# ============================================================================

print("PROCESSED PRODUCTS - Company X")
print("="*60)
for p in products_x:
    discount_info = f" ({p['discount_pct']}% off)" if p['discount_pct'] > 0 else ""
    print(f"  {p['product_id']}: {p['product_name']}")
    print(f"    Price: ${p['effective_price']}{discount_info}")
    print(f"    Features: {p['feature_count']} | Value: ${p['price_per_feature']}/feature")
    print()

print("\nPROCESSED PRODUCTS - Company Y")
print("="*60)
for p in products_y:
    discount_info = f" ({p['discount_pct']}% off)" if p['discount_pct'] > 0 else ""
    print(f"  {p['product_id']}: {p['product_name']}")
    print(f"    Price: ${p['effective_price']}{discount_info}")
    print(f"    Features: {p['feature_count']} | Value: ${p['price_per_feature']}/feature")
    print()

## 2.2 Product Comparison Examples

Let's use our processor to compare products across categories.

In [ ]:
# ============================================================================
# EXAMPLE: HEAD-TO-HEAD COMPARISONS
# ============================================================================

print("HEAD-TO-HEAD PRODUCT COMPARISONS")
print("="*60)

# Compare products in each category
categories = set(p['category'] for p in all_products)

for category in categories:
    print(f"\nCategory: {category}")
    print("-"*50)

    cat_x = [p for p in products_x if p['category'] == category]
    cat_y = [p for p in products_y if p['category'] == category]

    # Compare first product from each
    if cat_x and cat_y:
        comparison = processor.compare_products(cat_x[0], cat_y[0])

        print(f"  {comparison['product_x']} vs {comparison['product_y']}")
        print(f"  Price: ${comparison['price_x']} vs ${comparison['price_y']} (Advantage: {comparison['price_advantage']})")
        print(f"  Features: {comparison['features_x']} vs {comparison['features_y']} (Advantage: {comparison['feature_advantage']})")
        print(f"  Value Score: {comparison['value_x']} vs {comparison['value_y']} (Advantage: {comparison['value_advantage']})")

        if comparison['unique_to_x']:
            print(f"  Unique to X: {', '.join(comparison['unique_to_x'][:3])}")
        if comparison['unique_to_y']:
            print(f"  Unique to Y: {', '.join(comparison['unique_to_y'][:3])}")

## Key Concept: Value Score Calculation

The **Value Score** is a key metric for competitive positioning:

```
Value Score = (Feature Count / Effective Price) * 100
```

### Interpretation

| Value Score | Interpretation |
|-------------|----------------|
| > 5.0 | Excellent value - high features, low price |
| 3.0 - 5.0 | Good value - balanced |
| < 3.0 | Premium positioning - fewer features per dollar |

### Business Implications

- **High value score**: Good for value-conscious marketing
- **Low value score**: Position on quality, brand, experience
- **Matching competitor**: Focus on unique features

## Checkpoint: Week 2 Verification

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT
# ============================================================================

print("WEEK 2 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Processor initialized", processor is not None),
    ("Company X products processed", len(products_x) > 0),
    ("Company Y products processed", len(products_y) > 0),
    ("All products have product_id", all('product_id' in p for p in all_products)),
    ("All products have effective_price", all('effective_price' in p for p in all_products)),
    ("Feature vocabulary built", len(processor.feature_vocabulary) > 0),
    ("Comparison function works", 'compare_products' in dir(processor)),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 2 Complete! You may proceed to Week 3.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 2 Summary

### What We Accomplished

| Component | Status | Key Outputs |
|-----------|--------|-------------|
| Catalog Processor | Complete | Normalized products |
| Discount Parsing | Complete | Effective prices |
| Feature Normalization | Complete | Vocabulary built |
| Product Comparison | Complete | Head-to-head analysis |

### Key Metrics Calculated

- Effective prices (after discounts)
- Feature counts
- Value scores (features per dollar)
- Price advantages

### Next Steps (Week 3)

- Create product embeddings
- Build ChromaDB vector store
- Enable semantic product search

---
# WEEK 3: Vector Store & Semantic Search (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 3: Vector Store & Semantic Search

---

## Learning Objectives

By the end of Week 3, you will be able to:

- [ ] Create product embeddings with Langfuse tracing
- [ ] Build a ChromaDB vector store
- [ ] Perform semantic product search
- [ ] Filter searches by company or category
- [ ] Understand embedding similarity metrics

## Deep Dive: Semantic Search in E-Commerce

Traditional search matches keywords. Semantic search understands **meaning**.

### Example

Query: "headphones for running"

| Search Type | Matches | Why |
|-------------|---------|-----|
| Keyword | Products with "running" in name | Literal match |
| Semantic | Waterproof, secure-fit earbuds | Understands intent |

### How It Works

1. **Embedding**: Convert product descriptions to vectors
2. **Indexing**: Store vectors in ChromaDB
3. **Query**: Convert search query to vector
4. **Similarity**: Find nearest vectors (cosine similarity)

### Why This Matters for Competitive Intelligence

- Find **similar competitor products** automatically
- Match products by **functionality**, not just name
- Enable **natural language queries** from business users

In [ ]:
# ============================================================================
# WEEK 3.1: CHROMADB VECTOR STORE CLASS
# ============================================================================

import chromadb

class TracedProductVectorStore:
    """
    ChromaDB vector store with full Langfuse tracing.

    Every embedding and search operation is logged for:
    - Cost tracking
    - Performance monitoring
    - Debug and audit
    """

    def __init__(self, persist_directory: str = "./chroma_ecommerce_db"):
        """
        Initialize ChromaDB with persistent storage.

        Args:
            persist_directory: Where to store the vector database
        """
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collections = {}
        print(f"ChromaDB initialized at: {persist_directory}")

    def product_to_text(self, product: Dict) -> str:
        """
        Convert product to searchable text representation.

        This text is what gets embedded for semantic search.
        """
        features_str = ', '.join(product.get('features', []))

        return f"""{product['category']} - {product['product_name']} by {product['company']}.
Price: ${product['effective_price']} (was ${product['base_price']}, {product.get('discount_text', 'no discount')}).
Features: {features_str}.
Availability: {product['availability']}.
Value: ${product['price_per_feature']} per feature."""

    def create_collection(self, name: str) -> chromadb.Collection:
        """
        Create or get a collection for products.

        Uses cosine similarity for comparing embeddings.
        """
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}
        )
        self.collections[name] = collection
        return collection

print("TracedProductVectorStore class defined")
print("  - product_to_text()")
print("  - create_collection()")

In [ ]:
# ============================================================================
# VECTOR STORE METHODS - Add and Search
# ============================================================================

class TracedProductVectorStore(TracedProductVectorStore):

    def add_products_with_tracing(self, collection_name: str, products: List[Dict]):
        """
        Add products to vector store with full Langfuse tracing.

        Each product gets:
        - Converted to text
        - Embedded via OpenAI
        - Stored with metadata
        """
        trace = langfuse.trace(
            name=f"index-products-{collection_name}",
            session_id=SESSION_ID,
            metadata={"collection": collection_name, "product_count": len(products)},
            tags=["indexing", "embeddings", "week-3"]
        )

        collection = self.create_collection(collection_name)

        ids = []
        embeddings = []
        documents = []
        metadatas = []

        for i, product in enumerate(products):
            span = trace.span(name=f"embed-{product['product_id']}")

            # Convert to text
            text = self.product_to_text(product)

            # Generate embedding with tracing
            embedding = traced_embedding(
                text,
                trace_name=f"embed-product-{product['product_id']}",
                metadata={"product": product['product_name']}
            )

            ids.append(product['product_id'])
            embeddings.append(embedding)
            documents.append(text)
            metadatas.append({
                'company': product['company'],
                'category': product['category'],
                'product_name': product['product_name'],
                'price': product['effective_price'],
                'feature_count': product['feature_count']
            })

            span.end(output={"text_length": len(text)})
            print(f"  Embedded: {product['product_name']}")

        # Add to collection
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas
        )

        trace.update(output={"products_indexed": len(ids)})
        print(f"\nAdded {len(ids)} products to '{collection_name}'")

    def search_with_tracing(
        self,
        query: str,
        n_results: int = 5,
        company_filter: str = None,
        category_filter: str = None
    ) -> Dict:
        """
        Semantic search with Langfuse tracing.

        Args:
            query: Natural language search query
            n_results: Number of results to return
            company_filter: Filter by company name
            category_filter: Filter by product category

        Returns:
            Search results with documents, metadata, and distances
        """
        trace = langfuse.trace(
            name="product-search",
            session_id=SESSION_ID,
            input={
                "query": query,
                "n_results": n_results,
                "company_filter": company_filter,
                "category_filter": category_filter
            },
            tags=["search", "week-3"]
        )

        collection = self.collections.get('products')
        if not collection:
            return {'error': 'Collection not found'}

        # Embed query
        query_embedding = traced_embedding(query, "search-query-embedding")

        # Build filter
        where_filter = None
        if company_filter and category_filter:
            where_filter = {
                "$and": [
                    {"company": company_filter},
                    {"category": category_filter}
                ]
            }
        elif company_filter:
            where_filter = {"company": company_filter}
        elif category_filter:
            where_filter = {"category": category_filter}

        # Search
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_filter,
            include=['documents', 'metadatas', 'distances']
        )

        result_count = len(results['documents'][0]) if results['documents'] else 0
        trace.update(output={"results_count": result_count})

        return results

print("Additional methods added:")
print("  - add_products_with_tracing()")
print("  - search_with_tracing()")

In [ ]:
# ============================================================================
# INITIALIZE AND POPULATE VECTOR STORE
# ============================================================================

print("Initializing Product Vector Store...")
print("="*50)

# Initialize store
product_vector_store = TracedProductVectorStore()

# Add all products with tracing
print("\nAdding products to vector store:")
product_vector_store.add_products_with_tracing('products', all_products)

## 3.2 Semantic Search Examples

Let's test our semantic search with various queries. Each query is traced to Langfuse.

In [ ]:
# ============================================================================
# EXAMPLE: BASIC SEMANTIC SEARCH
# ============================================================================

print("SEMANTIC PRODUCT SEARCH - Basic Queries")
print("="*60)

basic_queries = [
    "noise cancelling headphones with long battery",
    "affordable waterproof speaker",
    "fitness watch with heart rate monitor",
]

for query in basic_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)

    results = product_vector_store.search_with_tracing(query, n_results=3)

    if results.get('documents') and results['documents'][0]:
        for i, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ), 1):
            similarity = 1 - dist
            print(f"  {i}. {meta['product_name']} ({meta['company']})")
            print(f"     Price: ${meta['price']} | Features: {meta['feature_count']}")
            print(f"     Similarity: {similarity:.3f}")

In [ ]:
# ============================================================================
# EXAMPLE: FILTERED SEARCH BY COMPANY
# ============================================================================

print("\nFILTERED SEARCH - By Company")
print("="*60)

query = "premium wireless audio"

for company in ['Company X', 'Company Y']:
    print(f"\nQuery: '{query}' (Company: {company})")
    print("-" * 50)

    results = product_vector_store.search_with_tracing(
        query,
        n_results=2,
        company_filter=company
    )

    if results.get('documents') and results['documents'][0]:
        for i, (meta, dist) in enumerate(zip(
            results['metadatas'][0],
            results['distances'][0]
        ), 1):
            similarity = 1 - dist
            print(f"  {i}. {meta['product_name']} - ${meta['price']} (Sim: {similarity:.3f})")

In [ ]:
# ============================================================================
# EXAMPLE: FILTERED SEARCH BY CATEGORY
# ============================================================================

print("\nFILTERED SEARCH - By Category")
print("="*60)

category_queries = [
    ("Wireless Headphones", "best value headphones"),
    ("Smart Watches", "advanced health tracking"),
    ("Portable Speakers", "party speaker with good bass"),
]

for category, query in category_queries:
    print(f"\nCategory: {category}")
    print(f"Query: '{query}'")
    print("-" * 40)

    results = product_vector_store.search_with_tracing(
        query,
        n_results=2,
        category_filter=category
    )

    if results.get('documents') and results['documents'][0]:
        for i, meta in enumerate(results['metadatas'][0], 1):
            print(f"  {i}. {meta['product_name']} ({meta['company']}) - ${meta['price']}")

## Checkpoint: Week 3 Verification

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT
# ============================================================================

print("WEEK 3 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Vector store initialized", product_vector_store is not None),
    ("Products collection created", 'products' in product_vector_store.collections),
    ("Search function works", hasattr(product_vector_store, 'search_with_tracing')),
    ("Filter by company works", True),  # Tested above
    ("Filter by category works", True),  # Tested above
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 3 Complete! You may proceed to Week 4.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 3 Summary

### What We Accomplished

| Component | Status | Traces |
|-----------|--------|--------|
| ChromaDB Setup | Complete | index-products-* |
| Product Embeddings | Complete | embed-product-* |
| Semantic Search | Complete | product-search |
| Filtered Search | Complete | With company/category |

### Key Capabilities

- Natural language product queries
- Company-specific searches
- Category filtering
- Similarity scoring

### Next Steps (Week 4)

- Design Price Monitor Agent
- Design Catalog Analyzer Agent
- Design Marketing Agent

---
# WEEK 4: Single Agent Design (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 4: Single Agent Design

---

## Learning Objectives

By the end of Week 4, you will be able to:

- [ ] Design a Price Monitor Agent with structured outputs
- [ ] Design a Catalog Analyzer Agent for feature comparison
- [ ] Design a Marketing Agent for content generation
- [ ] Use Pydantic models for type-safe agent outputs
- [ ] Implement full Langfuse tracing for each agent

## Deep Dive: Agent Architecture for E-Commerce Intelligence

Our competitive intelligence system uses specialized agents, each with a focused responsibility:

### Agent Responsibilities

| Agent | Input | Output | Business Value |
|-------|-------|--------|----------------|
| **Price Monitor** | Product catalogs | Pricing position, recommendations | Revenue optimization |
| **Catalog Analyzer** | Feature lists | Feature gaps, advantages | Product strategy |
| **Marketing Agent** | Product + competitor data | Marketing copy, claims | Customer acquisition |

### Why Structured Outputs?

Using Pydantic models for agent outputs provides:

1. **Type Safety**: Catch errors before they cause problems
2. **Consistency**: Every agent output has the same structure
3. **Integration**: Easy to pipe outputs between agents
4. **Validation**: Automatic validation of LLM responses

### Agent Design Pattern

```python
class Agent:
    def __init__(self):
        self.prompt = ChatPromptTemplate(...)  # Define behavior
        self.parser = PydanticOutputParser(...)  # Structure output
    
    def analyze(self, data):
        handler = get_langfuse_handler(...)  # Trace everything
        chain = self.prompt | llm | self.parser
        return chain.invoke(data)
```

In [ ]:
# ============================================================================
# WEEK 4.1: PYDANTIC MODELS FOR STRUCTURED OUTPUTS
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class PriceAnalysis(BaseModel):
    """Structured output for price analysis."""
    category: str = Field(description="Product category analyzed")
    our_avg_price: float = Field(description="Our average effective price")
    competitor_avg_price: float = Field(description="Competitor average effective price")
    price_position: str = Field(description="PREMIUM, COMPETITIVE, or VALUE")
    price_gap_pct: float = Field(description="Price gap as percentage")
    recommendations: List[str] = Field(description="List of pricing recommendations")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")

print("PriceAnalysis model defined")
print(f"  Fields: {list(PriceAnalysis.model_fields.keys())}")

In [ ]:
# ============================================================================
# FEATURE ANALYSIS MODEL
# ============================================================================

class FeatureAnalysis(BaseModel):
    """Structured output for feature/catalog analysis."""
    category: str = Field(description="Product category analyzed")
    our_strengths: List[str] = Field(description="Features unique to our products")
    competitor_strengths: List[str] = Field(description="Features unique to competitor")
    feature_gaps: List[str] = Field(description="Features we should consider adding")
    competitive_advantage: str = Field(description="Our main competitive advantage")
    recommendations: List[str] = Field(description="Product development recommendations")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")

print("FeatureAnalysis model defined")
print(f"  Fields: {list(FeatureAnalysis.model_fields.keys())}")

In [ ]:
# ============================================================================
# MARKETING CONTENT MODEL
# ============================================================================

class MarketingContent(BaseModel):
    """Structured output for marketing content generation."""
    product_name: str = Field(description="Product being marketed")
    headline: str = Field(description="Attention-grabbing marketing headline")
    key_benefits: List[str] = Field(description="Top benefits to highlight")
    competitive_claims: List[str] = Field(description="Defensible claims vs competitors")
    target_audience: str = Field(description="Target customer segment")
    call_to_action: str = Field(description="Compelling CTA")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")

print("MarketingContent model defined")
print(f"  Fields: {list(MarketingContent.model_fields.keys())}")

## Key Concept: Pydantic for LLM Outputs

Pydantic models serve as a **contract** between our prompts and code:

### Field Definitions

```python
class PriceAnalysis(BaseModel):
    price_position: str = Field(
        description="PREMIUM, COMPETITIVE, or VALUE"  # This goes to the LLM!
    )
    confidence: float = Field(
        ge=0.0, le=1.0,  # Validation constraint
        description="Confidence score 0-1"
    )
```

### Benefits

1. **Descriptions guide the LLM**: The description field tells the model what to produce
2. **Constraints validate output**: `ge=0.0, le=1.0` ensures confidence is in range
3. **Type hints enable IDE support**: Your IDE knows what fields are available
4. **JSON serialization built-in**: Easy to log, store, or transmit

In [ ]:
# ============================================================================
# WEEK 4.2: PRICE MONITOR AGENT
# ============================================================================

class TracedPriceMonitorAgent:
    """
    Price Monitor Agent with full Langfuse tracing.

    Analyzes competitor pricing and provides strategic recommendations.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=PriceAnalysis)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a pricing strategy expert for e-commerce.
Analyze competitor pricing and provide actionable recommendations.

Consider:
- Price elasticity and market positioning
- Value perception vs actual features
- Discount strategies and their effectiveness
- Seasonal and promotional timing

Determine price position:
- PREMIUM: Our price > competitor by 10%+
- COMPETITIVE: Within +/- 10% of competitor
- VALUE: Our price < competitor by 10%+

{format_instructions}"""),
            ("human", "Analyze pricing for this category:\n\n{product_data}")
        ])

    def analyze(self, product_data: str, category: str = "unknown") -> Dict:
        """
        Analyze pricing with Langfuse tracing.

        Args:
            product_data: Formatted product comparison data
            category: Product category being analyzed

        Returns:
            Dict with success status and PriceAnalysis result
        """
        handler = get_langfuse_handler(
            trace_name=f"price-analysis-{category.lower().replace(' ', '-')}",
            tags=["price", "agent", "analysis", "week-4"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "product_data": product_data,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

# Initialize agent
price_agent = TracedPriceMonitorAgent()
print("TracedPriceMonitorAgent initialized")

In [ ]:
# ============================================================================
# CATALOG ANALYZER AGENT
# ============================================================================

class TracedCatalogAnalyzerAgent:
    """
    Catalog Analyzer Agent with full Langfuse tracing.

    Analyzes product features and competitive positioning.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=FeatureAnalysis)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a product strategy expert.
Analyze product features and competitive positioning.

Consider:
- Feature parity and gaps
- Unique selling propositions
- Market trends and customer expectations
- Technical specifications importance

Focus on actionable insights that can drive product decisions.

{format_instructions}"""),
            ("human", "Analyze features for this category:\n\n{product_data}")
        ])

    def analyze(self, product_data: str, category: str = "unknown") -> Dict:
        """Analyze features with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"feature-analysis-{category.lower().replace(' ', '-')}",
            tags=["feature", "agent", "analysis", "week-4"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "product_data": product_data,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

# Initialize agent
catalog_agent = TracedCatalogAnalyzerAgent()
print("TracedCatalogAnalyzerAgent initialized")

In [ ]:
# ============================================================================
# MARKETING AGENT
# ============================================================================

class TracedMarketingAgent:
    """
    Marketing Agent with full Langfuse tracing.

    Generates competitive marketing content.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=MarketingContent)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a marketing expert for consumer electronics.
Create compelling marketing content that highlights competitive advantages.

Guidelines:
- Focus on customer benefits, not just features
- Make credible, defensible competitive claims
- Use engaging, action-oriented language
- Target the right customer segment
- Include a clear call to action

{format_instructions}"""),
            ("human", "Create marketing content for:\n\nOur Product:\n{product_data}\n\nCompetitor:\n{competitor_data}")
        ])

    def generate(self, product_data: str, competitor_data: str, product_name: str = "product") -> Dict:
        """Generate marketing content with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"marketing-{product_name.lower().replace(' ', '-')}",
            tags=["marketing", "agent", "content", "week-4"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0.7, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "product_data": product_data,
                "competitor_data": competitor_data,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

# Initialize agent
marketing_agent = TracedMarketingAgent()
print("TracedMarketingAgent initialized")

print("\nAll agents ready:")
print("  - price_agent (TracedPriceMonitorAgent)")
print("  - catalog_agent (TracedCatalogAnalyzerAgent)")
print("  - marketing_agent (TracedMarketingAgent)")

## 4.3 Agent Examples

Let's run each agent with sample data. All invocations are traced to Langfuse.

In [ ]:
# ============================================================================
# EXAMPLE: PRICE MONITOR AGENT - Multiple Categories
# ============================================================================

print("PRICE MONITOR AGENT - Analysis Examples")
print("="*60)

categories = ["Wireless Headphones", "Smart Watches", "Portable Speakers"]

for category in categories:
    print(f"\n[Category: {category}]")
    print("-" * 40)

    # Prepare data for this category
    x_prods = [p for p in products_x if p['category'] == category]
    y_prods = [p for p in products_y if p['category'] == category]

    data_text = f"Our Products (Company X):\n"
    for p in x_prods:
        data_text += f"- {p['product_name']}: ${p['effective_price']}, Features: {', '.join(p['features'])}\n"
    data_text += f"\nCompetitor Products (Company Y):\n"
    for p in y_prods:
        data_text += f"- {p['product_name']}: ${p['effective_price']}, Features: {', '.join(p['features'])}\n"

    result = price_agent.analyze(data_text, category)

    if result['success']:
        r = result['result']
        print(f"  Position: {r['price_position']}")
        print(f"  Our Avg: ${r['our_avg_price']:.2f}")
        print(f"  Competitor Avg: ${r['competitor_avg_price']:.2f}")
        print(f"  Gap: {r['price_gap_pct']:.1f}%")
        print(f"  Confidence: {r['confidence']:.2f}")
        print(f"  Top Recommendation: {r['recommendations'][0] if r['recommendations'] else 'N/A'}")
    else:
        print(f"  Error: {result['error']}")

In [ ]:
# ============================================================================
# EXAMPLE: CATALOG ANALYZER AGENT
# ============================================================================

print("\nCATALOG ANALYZER AGENT - Feature Analysis")
print("="*60)

for category in categories:
    print(f"\n[Category: {category}]")
    print("-" * 40)

    x_prods = [p for p in products_x if p['category'] == category]
    y_prods = [p for p in products_y if p['category'] == category]

    data_text = f"Our Products (Company X):\n"
    for p in x_prods:
        data_text += f"- {p['product_name']}: Features: {', '.join(p['features'])}\n"
    data_text += f"\nCompetitor Products (Company Y):\n"
    for p in y_prods:
        data_text += f"- {p['product_name']}: Features: {', '.join(p['features'])}\n"

    result = catalog_agent.analyze(data_text, category)

    if result['success']:
        r = result['result']
        print(f"  Our Strengths: {r['our_strengths'][:2]}")
        print(f"  Competitor Strengths: {r['competitor_strengths'][:2]}")
        print(f"  Feature Gaps: {r['feature_gaps'][:2] if r['feature_gaps'] else 'None identified'}")
        print(f"  Competitive Advantage: {r['competitive_advantage']}")
    else:
        print(f"  Error: {result['error']}")

In [ ]:
# ============================================================================
# EXAMPLE: MARKETING AGENT - Content Generation
# ============================================================================

print("\nMARKETING AGENT - Content Generation")
print("="*60)

# Generate marketing for select products
for product in products_x[:3]:
    print(f"\n[Product: {product['product_name']}]")
    print("-" * 40)

    # Find competitor in same category
    competitors = [p for p in products_y if p['category'] == product['category']]
    competitor = competitors[0] if competitors else None

    product_text = f"{product['product_name']}: ${product['effective_price']}, Features: {', '.join(product['features'])}"
    competitor_text = f"{competitor['product_name']}: ${competitor['effective_price']}, Features: {', '.join(competitor['features'])}" if competitor else "No direct competitor"

    result = marketing_agent.generate(product_text, competitor_text, product['product_name'])

    if result['success']:
        r = result['result']
        print(f"  Headline: {r['headline']}")
        print(f"  Target: {r['target_audience']}")
        print(f"  Key Benefits: {r['key_benefits'][:2]}")
        print(f"  CTA: {r['call_to_action']}")
    else:
        print(f"  Error: {result['error']}")

In [ ]:
# ============================================================================
# WEEK 4 CHECKPOINT
# ============================================================================

print("WEEK 4 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("PriceAnalysis model defined", 'PriceAnalysis' in globals()),
    ("FeatureAnalysis model defined", 'FeatureAnalysis' in globals()),
    ("MarketingContent model defined", 'MarketingContent' in globals()),
    ("Price agent initialized", price_agent is not None),
    ("Catalog agent initialized", catalog_agent is not None),
    ("Marketing agent initialized", marketing_agent is not None),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 4 Complete! You may proceed to Week 5.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 4 Summary

### What We Accomplished

| Component | Status | Traces |
|-----------|--------|--------|
| Pydantic Models | Complete | N/A |
| Price Monitor Agent | Complete | price-analysis-* |
| Catalog Analyzer Agent | Complete | feature-analysis-* |
| Marketing Agent | Complete | marketing-* |

### Agent Capabilities

- **Price Agent**: Pricing position analysis, gap identification, recommendations
- **Catalog Agent**: Feature comparison, gap analysis, competitive advantages
- **Marketing Agent**: Headlines, benefits, competitive claims, CTAs

### Next Steps (Week 5)

- Coordinate multiple agents
- Implement parallel analysis
- Aggregate insights across agents

---
# WEEK 5: Multi-Agent Orchestration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 5: Multi-Agent Orchestration

---

## Learning Objectives

By the end of Week 5, you will be able to:

- [ ] Coordinate multiple agents in a workflow
- [ ] Implement parallel agent execution
- [ ] Aggregate insights from multiple agents
- [ ] Build an orchestration layer with full tracing

## Deep Dive: Multi-Agent Orchestration Patterns

Complex business problems require multiple AI agents working together. There are several orchestration patterns:

### Pattern 1: Sequential Pipeline

```
Price Agent -> Catalog Agent -> Marketing Agent
```

Each agent's output feeds into the next. Good for dependent analysis.

### Pattern 2: Parallel Analysis (Our Approach)

```
         +--> Price Agent ----+
Input -->+--> Catalog Agent --+--> Aggregator --> Output
         +--> Marketing Agent +
```

All agents analyze simultaneously. Good for comprehensive reports.

### Pattern 3: Supervisor/Worker

```
Supervisor Agent
    |
    +--> Delegate to Price Agent
    +--> Delegate to Catalog Agent
    +--> Synthesize results
```

A supervisor agent decides which workers to invoke.

### Why Parallel Pattern for E-Commerce?

- **Speed**: Don't wait for sequential completion
- **Independence**: Price, features, and marketing are orthogonal
- **Completeness**: Always get all three perspectives

In [ ]:
# ============================================================================
# WEEK 5.1: COMPETITIVE INTELLIGENCE ORCHESTRATOR
# ============================================================================

import concurrent.futures
from tenacity import retry, stop_after_attempt, wait_exponential

class TracedCompetitiveIntelligenceOrchestrator:
    """
    Multi-agent orchestrator with full Langfuse tracing.

    Coordinates Price, Catalog, and Marketing agents for
    comprehensive competitive analysis.
    """

    def __init__(self):
        self.price_agent = price_agent
        self.catalog_agent = catalog_agent
        self.marketing_agent = marketing_agent

    def prepare_category_data(self, category: str) -> Dict:
        """Prepare analysis data for a specific category."""
        x_products = [p for p in products_x if p['category'] == category]
        y_products = [p for p in products_y if p['category'] == category]

        text = "Our Products (Company X):\n"
        for p in x_products:
            text += f"- {p['product_name']}: ${p['effective_price']}, Features: {', '.join(p['features'])}\n"
        text += "\nCompetitor Products (Company Y):\n"
        for p in y_products:
            text += f"- {p['product_name']}: ${p['effective_price']}, Features: {', '.join(p['features'])}\n"

        return {
            'category': category,
            'company_x': x_products,
            'company_y': y_products,
            'text': text
        }

print("TracedCompetitiveIntelligenceOrchestrator base defined")

In [ ]:
# ============================================================================
# ORCHESTRATOR - Analysis Methods
# ============================================================================

class TracedCompetitiveIntelligenceOrchestrator(TracedCompetitiveIntelligenceOrchestrator):

    def analyze_category_with_tracing(self, category: str) -> Dict:
        """
        Run full competitive analysis with orchestration tracing.

        Executes price and feature analysis in parallel,
        then aggregates insights.
        """
        trace = langfuse.trace(
            name=f"orchestration-{category.lower().replace(' ', '-')}",
            session_id=SESSION_ID,
            input={"category": category},
            tags=["orchestration", "multi-agent", "week-5"]
        )

        data = self.prepare_category_data(category)

        results = {
            'category': category,
            'timestamp': datetime.now().isoformat(),
            'analyses': {}
        }

        # Run price and feature analysis in parallel
        parallel_span = trace.span(name="parallel-analysis")

        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            price_future = executor.submit(
                self.price_agent.analyze, data['text'], category
            )
            feature_future = executor.submit(
                self.catalog_agent.analyze, data['text'], category
            )

            results['analyses']['pricing'] = price_future.result()
            results['analyses']['features'] = feature_future.result()

        parallel_span.end(output={
            "price_success": results['analyses']['pricing']['success'],
            "feature_success": results['analyses']['features']['success']
        })

        # Aggregate insights
        agg_span = trace.span(name="aggregate-insights")
        results['summary'] = self._aggregate_insights(results['analyses'])
        agg_span.end(output=results['summary'])

        trace.update(output=results['summary'])

        return results

    def _aggregate_insights(self, analyses: Dict) -> Dict:
        """Aggregate insights from all agents into a summary."""
        summary = {
            'key_findings': [],
            'priority_actions': [],
            'overall_position': 'UNKNOWN'
        }

        # Extract from pricing analysis
        if analyses.get('pricing', {}).get('success'):
            price_result = analyses['pricing']['result']
            summary['key_findings'].append(
                f"Price Position: {price_result.get('price_position', 'N/A')}"
            )
            summary['priority_actions'].extend(
                price_result.get('recommendations', [])[:2]
            )
            summary['overall_position'] = price_result.get('price_position', 'UNKNOWN')

        # Extract from feature analysis
        if analyses.get('features', {}).get('success'):
            feature_result = analyses['features']['result']
            summary['key_findings'].append(
                f"Competitive Advantage: {feature_result.get('competitive_advantage', 'N/A')}"
            )
            if feature_result.get('feature_gaps'):
                summary['priority_actions'].append(
                    f"Address feature gap: {feature_result['feature_gaps'][0]}"
                )

        return summary

# Initialize orchestrator
ci_orchestrator = TracedCompetitiveIntelligenceOrchestrator()
print("TracedCompetitiveIntelligenceOrchestrator initialized with parallel analysis")

In [ ]:
# ============================================================================
# EXAMPLE: ORCHESTRATED ANALYSIS - All Categories
# ============================================================================

print("ORCHESTRATED COMPETITIVE ANALYSIS")
print("="*60)
print("Running parallel agents for each category...\n")

categories = ["Wireless Headphones", "Smart Watches", "Portable Speakers"]

all_results = []
for category in categories:
    print(f"[Orchestration: {category}]")
    print("-" * 50)

    result = ci_orchestrator.analyze_category_with_tracing(category)
    all_results.append(result)

    print(f"  Key Findings:")
    for finding in result['summary']['key_findings']:
        print(f"    - {finding}")

    print(f"  Priority Actions:")
    for action in result['summary']['priority_actions'][:3]:
        print(f"    - {action}")

    print(f"  Overall Position: {result['summary']['overall_position']}")
    print()

print(f"\nTotal orchestrations completed: {len(all_results)}")

In [ ]:
# ============================================================================
# WEEK 5 CHECKPOINT
# ============================================================================

print("WEEK 5 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Orchestrator initialized", ci_orchestrator is not None),
    ("Parallel analysis works", len(all_results) > 0),
    ("Results have summaries", all('summary' in r for r in all_results)),
    ("Key findings extracted", all(len(r['summary']['key_findings']) > 0 for r in all_results)),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 5 Complete! You may proceed to Week 6.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 5 Summary

### What We Accomplished

| Component | Status | Traces |
|-----------|--------|--------|
| Orchestrator Class | Complete | orchestration-* |
| Parallel Execution | Complete | parallel-analysis spans |
| Insight Aggregation | Complete | aggregate-insights spans |

### Key Capabilities

- Multi-agent coordination
- Parallel execution for speed
- Unified insights from multiple perspectives

### Next Steps (Week 6)

- Build product knowledge graph
- Visualize competitive relationships
- Enable graph-based reasoning

---
# WEEK 6: Product Knowledge Graph (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 6: Product Knowledge Graph

---

## Learning Objectives

By the end of Week 6, you will be able to:

- [ ] Build a product knowledge graph using NetworkX
- [ ] Model relationships between products, features, and companies
- [ ] Visualize the competitive landscape
- [ ] Implement graph-based reasoning for competitive analysis

## Deep Dive: Knowledge Graphs in E-Commerce

Knowledge graphs represent information as **nodes** (entities) and **edges** (relationships).

### Our Graph Schema

```
NODES:
- COMPANY: Company X, Company Y
- CATEGORY: Wireless Headphones, Smart Watches, etc.
- PRODUCT: Individual products
- FEATURE: Bluetooth 5.0, Noise Cancelling, etc.

EDGES:
- SELLS: Company -> Product
- IN_CATEGORY: Product -> Category
- HAS_FEATURE: Product -> Feature
- COMPETES_WITH: Product -> Product
```

### Business Value

| Query Type | Example | Value |
|------------|---------|-------|
| Feature Overlap | "Products with Bluetooth 5.2" | Technology tracking |
| Competition | "Direct competitors to Watch X1" | Market mapping |
| Gaps | "Features in Y's products not in X's" | Product strategy |
| Clustering | "Products similar to Headphones X2" | Portfolio analysis |

In [ ]:
# ============================================================================
# WEEK 6.1: PRODUCT KNOWLEDGE GRAPH
# ============================================================================

import networkx as nx
from pyvis.network import Network

class ProductKnowledgeGraph:
    """
    Knowledge graph for product relationships and competitive landscape.
    """

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_product_taxonomy()

    def _build_product_taxonomy(self):
        """Build product category taxonomy."""
        # Add category nodes
        for cat in ['Wireless Headphones', 'Smart Watches', 'Portable Speakers']:
            self.graph.add_node(cat, type='CATEGORY')

        # Add company nodes
        for company in ['Company X', 'Company Y']:
            self.graph.add_node(company, type='COMPANY')

        # Add common feature nodes
        features = [
            'Bluetooth 5.0', 'Bluetooth 5.2', 'Noise Cancelling', 'Waterproof',
            'Long Battery', 'GPS', 'Heart Rate', 'Foldable', 'USB-C'
        ]
        for feature in features:
            self.graph.add_node(feature, type='FEATURE')

    def add_product(self, product: Dict):
        """Add a product and its relationships to the graph."""
        product_id = product['product_id']

        # Add product node
        self.graph.add_node(
            product_id,
            type='PRODUCT',
            name=product['product_name'],
            price=product['effective_price']
        )

        # Add relationships
        self.graph.add_edge(product['company'], product_id, relation='SELLS')
        self.graph.add_edge(product_id, product['category'], relation='IN_CATEGORY')

        # Add feature relationships
        for feature in product['features']:
            if not self.graph.has_node(feature):
                self.graph.add_node(feature, type='FEATURE')
            self.graph.add_edge(product_id, feature, relation='HAS_FEATURE')

print("ProductKnowledgeGraph class defined")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH - Query Methods
# ============================================================================

class ProductKnowledgeGraph(ProductKnowledgeGraph):

    def find_competing_products(self, product_id: str) -> List[str]:
        """Find products competing with the given product."""
        # Get product's category
        category = None
        for _, target, data in self.graph.out_edges(product_id, data=True):
            if data.get('relation') == 'IN_CATEGORY':
                category = target
                break

        if not category:
            return []

        # Find all products in same category
        competitors = []
        for source, _, data in self.graph.in_edges(category, data=True):
            if data.get('relation') == 'IN_CATEGORY' and source != product_id:
                competitors.append(source)

        return competitors

    def get_shared_features(self, product_id1: str, product_id2: str) -> List[str]:
        """Get features shared between two products."""
        features1 = set()
        features2 = set()

        for _, target, data in self.graph.out_edges(product_id1, data=True):
            if data.get('relation') == 'HAS_FEATURE':
                features1.add(target)

        for _, target, data in self.graph.out_edges(product_id2, data=True):
            if data.get('relation') == 'HAS_FEATURE':
                features2.add(target)

        return list(features1 & features2)

    def get_unique_features(self, product_id: str, competitor_id: str) -> Dict:
        """Get features unique to each product."""
        features_ours = set()
        features_theirs = set()

        for _, target, data in self.graph.out_edges(product_id, data=True):
            if data.get('relation') == 'HAS_FEATURE':
                features_ours.add(target)

        for _, target, data in self.graph.out_edges(competitor_id, data=True):
            if data.get('relation') == 'HAS_FEATURE':
                features_theirs.add(target)

        return {
            'unique_to_ours': list(features_ours - features_theirs),
            'unique_to_competitor': list(features_theirs - features_ours),
            'common': list(features_ours & features_theirs)
        }

print("Knowledge graph query methods added")

In [ ]:
# ============================================================================
# BUILD KNOWLEDGE GRAPH
# ============================================================================

print("Building Product Knowledge Graph...")
print("="*50)

# Initialize graph
product_kg = ProductKnowledgeGraph()

# Add all products
for product in all_products:
    product_kg.add_product(product)

print(f"\nKnowledge Graph Statistics:")
print(f"  Total Nodes: {product_kg.graph.number_of_nodes()}")
print(f"  Total Edges: {product_kg.graph.number_of_edges()}")

# Count by type
node_types = {}
for _, data in product_kg.graph.nodes(data=True):
    t = data.get('type', 'OTHER')
    node_types[t] = node_types.get(t, 0) + 1

print(f"\nNodes by Type:")
for t, count in sorted(node_types.items()):
    print(f"  {t}: {count}")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH VISUALIZATION
# ============================================================================

print("Creating Knowledge Graph Visualization...")

# Color map for node types
color_map = {
    'PRODUCT': '#3498db',
    'CATEGORY': '#e74c3c',
    'COMPANY': '#2ecc71',
    'FEATURE': '#f39c12'
}

# Create matplotlib visualization
node_colors = [
    color_map.get(product_kg.graph.nodes[n].get('type', 'OTHER'), '#95a5a6')
    for n in product_kg.graph.nodes()
]

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(product_kg.graph, k=2, iterations=50, seed=42)

nx.draw_networkx_nodes(product_kg.graph, pos, node_color=node_colors,
                       node_size=800, alpha=0.9)
nx.draw_networkx_edges(product_kg.graph, pos, edge_color='gray',
                       arrows=True, arrowsize=15, alpha=0.6)

# Create labels
labels = {}
for node in product_kg.graph.nodes():
    data = product_kg.graph.nodes[node]
    if data.get('type') == 'PRODUCT':
        labels[node] = data.get('name', node)[:12]
    else:
        labels[node] = str(node)[:12]

nx.draw_networkx_labels(product_kg.graph, pos, labels, font_size=8)

# Legend
legend_elements = [
    plt.scatter([], [], c=color, s=100, label=node_type)
    for node_type, color in color_map.items()
]
plt.legend(handles=legend_elements, loc='upper left', title='Node Types')

plt.title('E-Commerce Product Knowledge Graph', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('product_knowledge_graph.png', dpi=150, bbox_inches='tight')
plt.show()

print("Graph saved to product_knowledge_graph.png")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH QUERIES
# ============================================================================

print("KNOWLEDGE GRAPH COMPETITIVE ANALYSIS")
print("="*60)

# Log to Langfuse
kg_trace = langfuse.trace(
    name="knowledge-graph-analysis",
    session_id=SESSION_ID,
    tags=["knowledge-graph", "week-6"]
)

# Query 1: Find competitors
print("\n[Query 1] Finding Competitors")
print("-" * 40)
for product in products_x[:3]:
    competitors = product_kg.find_competing_products(product['product_id'])
    print(f"  {product['product_name']} competes with: {len(competitors)} products")
    for comp_id in competitors[:2]:
        comp_data = product_kg.graph.nodes[comp_id]
        print(f"    - {comp_data.get('name', comp_id)}")

# Query 2: Shared features
print("\n[Query 2] Feature Comparison")
print("-" * 40)
p1 = products_x[0]
p2 = products_y[0]
comparison = product_kg.get_unique_features(p1['product_id'], p2['product_id'])
print(f"  {p1['product_name']} vs {p2['product_name']}")
print(f"  Unique to ours: {comparison['unique_to_ours']}")
print(f"  Unique to competitor: {comparison['unique_to_competitor']}")
print(f"  Common: {comparison['common']}")

kg_trace.update(output={
    "total_nodes": product_kg.graph.number_of_nodes(),
    "total_edges": product_kg.graph.number_of_edges()
})

In [ ]:
# ============================================================================
# WEEK 6 CHECKPOINT
# ============================================================================

print("WEEK 6 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("Knowledge graph initialized", product_kg is not None),
    ("Products added to graph", product_kg.graph.number_of_nodes() > 10),
    ("Competitor query works", len(product_kg.find_competing_products(products_x[0]['product_id'])) > 0),
    ("Feature comparison works", 'common' in comparison),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("Week 6 Complete! You may proceed to Week 7.")
else:
    print("Some checks failed. Please review before continuing.")

## Week 6 Summary

### What We Accomplished

| Component | Status |
|-----------|--------|
| Knowledge Graph | Complete |
| Product Relationships | Complete |
| Feature Mapping | Complete |
| Graph Visualization | Complete |
| Competitive Queries | Complete |

### Next Steps (Week 7 & 8)

- Advanced observability metrics
- Production system integration
- Gradio UI deployment

---
# WEEK 7: Advanced Observability & System Integration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEKS 7 & 8: Production System & Deployment

---

## Learning Objectives

By the end of Weeks 7 & 8, you will be able to:

- [ ] Integrate all components into a production system
- [ ] Create an interactive Gradio interface
- [ ] Implement comprehensive error handling
- [ ] Deploy a fully traced competitive intelligence system

In [ ]:
# ============================================================================
# WEEKS 7 & 8: FULL SYSTEM INTEGRATION
# ============================================================================

class EcommerceIntelligenceSystem:
    """
    Production E-Commerce Competitive Intelligence System.

    Integrates all components:
    - Catalog processing
    - Vector search
    - Multi-agent orchestration
    - Knowledge graph
    - Full Langfuse tracing
    """

    def __init__(self):
        self.processor = processor
        self.vector_store = product_vector_store
        self.orchestrator = ci_orchestrator
        self.knowledge_graph = product_kg
        self.products_x = products_x
        self.products_y = products_y

    def analyze_category(self, category: str) -> Dict:
        """Run full competitive analysis for a category."""
        return self.orchestrator.analyze_category_with_tracing(category)

    def search_products(self, query: str) -> Dict:
        """Semantic product search."""
        return self.vector_store.search_with_tracing(query, n_results=5)

    def get_price_comparison(self, category: str) -> pd.DataFrame:
        """Get price comparison for a category."""
        x_prods = [p for p in self.products_x if p['category'] == category]
        y_prods = [p for p in self.products_y if p['category'] == category]

        data = []
        for p in x_prods + y_prods:
            data.append({
                'Company': p['company'],
                'Product': p['product_name'],
                'Price': p['effective_price'],
                'Features': p['feature_count']
            })

        return pd.DataFrame(data)

    def get_status(self) -> Dict:
        """Get system status."""
        return {
            'session_id': SESSION_ID,
            'total_products': len(all_products),
            'kg_nodes': self.knowledge_graph.graph.number_of_nodes(),
            'kg_edges': self.knowledge_graph.graph.number_of_edges()
        }

# Initialize system
ecommerce_system = EcommerceIntelligenceSystem()

print("E-COMMERCE INTELLIGENCE SYSTEM")
print("="*50)
status = ecommerce_system.get_status()
print(f"Session: {status['session_id']}")
print(f"Products tracked: {status['total_products']}")
print(f"Knowledge Graph: {status['kg_nodes']} nodes, {status['kg_edges']} edges")

In [ ]:
# ============================================================================
# GRADIO INTERFACE
# ============================================================================

import gradio as gr

def analyze_category_ui(category: str) -> str:
    """UI function for category analysis."""
    result = ecommerce_system.analyze_category(category)

    response = f"""## Competitive Analysis: {category}\n\n"""
    response += "### Key Findings\n"
    for finding in result.get('summary', {}).get('key_findings', []):
        response += f"- {finding}\n"

    response += "\n### Priority Actions\n"
    for action in result.get('summary', {}).get('priority_actions', []):
        response += f"- {action}\n"

    response += f"\n### Overall Position: {result.get('summary', {}).get('overall_position', 'Unknown')}\n"

    return response

def search_products_ui(query: str) -> str:
    """UI function for product search."""
    results = ecommerce_system.search_products(query)

    response = f"## Search Results for: {query}\n\n"

    if results.get('documents') and results['documents'][0]:
        for i, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ), 1):
            similarity = 1 - dist
            response += f"**{i}. {meta['product_name']}** ({meta['company']}) - ${meta['price']}\n"
            response += f"   Similarity: {similarity:.3f}\n\n"
    else:
        response += "No results found.\n"

    return response

print("Gradio UI functions defined")

In [ ]:
# ============================================================================
# GRADIO APP DEFINITION
# ============================================================================

with gr.Blocks(title="E-Commerce Intelligence") as demo:
    gr.Markdown("""# E-Commerce Competitive Intelligence System

Multi-agent AI system for competitive analysis with full Langfuse observability.""")

    with gr.Tab("Category Analysis"):
        gr.Markdown("Analyze competitive positioning by product category.")
        category_dropdown = gr.Dropdown(
            choices=["Wireless Headphones", "Smart Watches", "Portable Speakers"],
            label="Select Category"
        )
        analyze_btn = gr.Button("Analyze", variant="primary")
        analysis_output = gr.Markdown()
        analyze_btn.click(analyze_category_ui, category_dropdown, analysis_output)

    with gr.Tab("Product Search"):
        gr.Markdown("Search products using natural language.")
        search_input = gr.Textbox(
            label="Search Query",
            placeholder="e.g., noise cancelling headphones with long battery..."
        )
        search_btn = gr.Button("Search", variant="primary")
        search_output = gr.Markdown()
        search_btn.click(search_products_ui, search_input, search_output)

    with gr.Tab("System Status"):
        gr.Markdown(f"""### System Information

- **Session ID**: {SESSION_ID}
- **Total Products**: {len(all_products)}
- **Langfuse Dashboard**: [View Traces]({os.environ.get('LANGFUSE_HOST')}/project)
""")

print("Gradio interface defined.")
print("\nTo launch:")
print("  demo.launch(share=True)  # In Colab")
print("  demo.launch()            # Locally")

In [ ]:
# ============================================================================
# LAUNCH GRADIO INTERFACE
# ============================================================================

if IN_COLAB:
    print("Launching Gradio interface with public URL...")
    demo.launch(share=True)
else:
    print("Launching Gradio interface locally...")
    print("Note: Set share=True for a public URL")
    # demo.launch()  # Uncomment to launch

---
# WEEK 8: Production Deployment & UI (YOUR WORK)
---

## Instructions

Complete the TODO cells below. Each cell has hints and instructions.
Refer to the learning objectives above and the master notebook for guidance.

---## Production Deployment: FastAPI ApplicationNow that we have a working Gradio prototype, let's create a **production-ready deployment** using:- **FastAPI** — Industry-standard async API framework- **Animated Web UI** — Beautiful interface showing the RAG pipeline flow in real-time- **Docker** — Containerized deployment ready for AWS/GCPThe web interface features animated visualizations of:- Document processing and chunking- Vector embedding and semantic search- Multi-agent orchestration and routing- Response generation pipeline---

In [ ]:
# ============================================================================
# TODO 8.1: PRODUCTION DEPLOYMENT: FastAPI Application
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement PRODUCTION DEPLOYMENT: FastAPI Application
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.2: ---------------------------------------------------------------------------
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def _lf_trace(name: str, **kwargs):
# HINT: def _lf_generation(trace, name: str, model: str, input_data):
# HINT: def _effective_price(p: dict) -> float:
# HINT: def _product_text(p: dict) -> str:
# HINT: def _init_vector_store():
# HINT: def _batch_embed(texts: List[str]) -> List[List[float]]:
# HINT: def _embed(text: str) -> List[float]:
# HINT: def _search(query: str, n: int = 5, where: dict = None) -> List[dict]:
# HINT: def _traced_completion(messages: List[dict], trace_name: str, model: str = "gpt-4o-mini") -> str:
# HINT: def _run_price_agent(category: str, context: str) -> dict:
# HINT: def _run_catalog_agent(category: str, context: str) -> dict:
# HINT: def _run_marketing_agent(query: str, context: str) -> dict:
# HINT: def run_pipeline(query: str) -> dict:
# HINT: class QueryRequest(BaseModel):
# HINT: class QueryResponse(BaseModel):
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement ---------------------------------------------------------------------------
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.3: Week 8 Task 3
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 3
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.4: Week 8 Task 4
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 4
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.5: Week 8 Task 5
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 5
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.6: Week 8 Task 6
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 6
raise NotImplementedError('Complete this section')

---

## Production Deployment: AWS EC2 Guide

After running the FastAPI cells above, you have a `deployment/` folder with everything needed. Here's how to deploy it to AWS EC2.

### Step 1: Launch an EC2 Instance

1. Go to **AWS Console → EC2 → Launch Instance**
2. **AMI**: Ubuntu Server 22.04 LTS (Free Tier eligible)
3. **Instance type**: `t2.micro` (Free Tier) for testing, `t3.small` or `t3.medium` for production
4. **Key pair**: Create or select an existing `.pem` key pair — download it
5. **Network settings**:
   - Allow SSH (port 22) from your IP
   - Add a **Custom TCP rule** for **port 8000** from `0.0.0.0/0` (or your IP for security)
6. **Storage**: 20 GB gp3 (default is fine)
7. Click **Launch Instance**

### Step 2: Transfer Files to EC2

**Option A: From your local machine (after downloading deployment/ from Colab)**

```bash
# Make your key file secure
chmod 400 your-key.pem

# Copy the deployment folder to EC2
scp -i your-key.pem -r deployment/ ubuntu@<EC2-PUBLIC-IP>:~/app/
```

**Option B: From Google Colab**

```python
# In Colab, zip the deployment folder
!zip -r deployment.zip deployment/

# Download it to your local machine
from google.colab import files
files.download('deployment.zip')

# Then scp from local to EC2 as shown in Option A
```

### Step 3: SSH into EC2 and Set Up

```bash
# Connect to your instance
ssh -i your-key.pem ubuntu@<EC2-PUBLIC-IP>

# Update system and install Python
sudo apt update && sudo apt upgrade -y
sudo apt install -y python3-pip python3-venv

# Set up the app
cd ~/app
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

### Step 4: Configure Environment Variables

```bash
# Create .env file with your API keys
cat > .env << 'EOF'
OPENAI_API_KEY=sk-your-openai-key
LANGFUSE_SECRET_KEY=sk-lf-your-secret
LANGFUSE_PUBLIC_KEY=pk-lf-your-public
LANGFUSE_HOST=https://cloud.langfuse.com
EOF
```

### Step 5: Launch the Application

```bash
# Test run (foreground)
uvicorn app:app --host 0.0.0.0 --port 8000

# Production run (background, survives SSH disconnect)
nohup uvicorn app:app --host 0.0.0.0 --port 8000 > app.log 2>&1 &

# Or use systemd for auto-restart:
sudo tee /etc/systemd/system/fastapi-app.service << 'EOF'
[Unit]
Description=FastAPI Competitive Intelligence App
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/app
Environment="PATH=/home/ubuntu/app/venv/bin"
EnvironmentFile=/home/ubuntu/app/.env
ExecStart=/home/ubuntu/app/venv/bin/uvicorn app:app --host 0.0.0.0 --port 8000
Restart=always

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl enable fastapi-app
sudo systemctl start fastapi-app
sudo systemctl status fastapi-app
```

### Step 6: Access Your Application

Open in your browser:
```
http://<EC2-PUBLIC-IP>:8000
```

API endpoints:
- **Health check**: `http://<EC2-PUBLIC-IP>:8000/api/health`
- **Query API**: `POST http://<EC2-PUBLIC-IP>:8000/api/query`
- **Swagger docs**: `http://<EC2-PUBLIC-IP>:8000/docs`

### Step 7: Verify Deployment

```bash
# From your local machine or any terminal
curl http://<EC2-PUBLIC-IP>:8000/api/health

# Test a query
curl -X POST http://<EC2-PUBLIC-IP>:8000/api/query \
  -H "Content-Type: application/json" \
  -d '{"query": "Analyze competitive pricing"}'
```

### Docker Alternative (Optional)

If you prefer Docker:

```bash
# On EC2
sudo apt install -y docker.io
sudo usermod -aG docker ubuntu
# Log out and back in, then:
cd ~/app
docker build -t intelligence-app .
docker run -d -p 8000:8000 --env-file .env --name app intelligence-app
```

### Security Recommendations for Production

- Use HTTPS with a reverse proxy (nginx + Let's Encrypt)
- Restrict port 8000 to your IP or use a load balancer
- Store API keys in AWS Secrets Manager instead of .env
- Set up CloudWatch monitoring for the EC2 instance
- Use an Elastic IP so the address doesn't change on reboot


In [ ]:
# ============================================================================
# TODO 8.7: DEPLOYMENT INSTRUCTIONS
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement DEPLOYMENT INSTRUCTIONS
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# FINAL CHECKPOINT - COMPLETE SYSTEM
# ============================================================================

print("FINAL CHECKPOINT - Complete System Verification")
print("="*60)

checks = [
    ("Langfuse tracing active", langfuse is not None),
    ("Product catalogs loaded", len(all_products) > 0),
    ("Vector store populated", 'products' in product_vector_store.collections),
    ("Agents initialized", all([price_agent, catalog_agent, marketing_agent])),
    ("Orchestrator ready", ci_orchestrator is not None),
    ("Knowledge graph built", product_kg.graph.number_of_nodes() > 0),
    ("Full system integrated", ecommerce_system is not None),
    ("Gradio interface defined", "demo" in globals()),
]

all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if not check_result:
        all_passed = False

print("\n" + "="*60)
if all_passed:
    print("CONGRATULATIONS! All 8 weeks completed successfully!")
    print("\nYour E-Commerce Intelligence System includes:")
    print("  - Full Langfuse observability")
    print("  - Semantic product search")
    print("  - Multi-agent competitive analysis")
    print("  - Product knowledge graph")
    print("  - Interactive Gradio UI")
else:
    print("Some components need attention.")

In [ ]:
# ============================================================================
# TODO 8.8: FINAL CLEANUP
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement FINAL CLEANUP
raise NotImplementedError('Complete this section')

## Course Summary

### 8-Week Journey Recap

| Week | Topic | Key Deliverable |
|------|-------|------------------|
| 1 | Environment & Data | Langfuse setup, EDA |
| 2 | Data Processing | Catalog normalization |
| 3 | Vector Store | Semantic search |
| 4 | Single Agents | Price, Catalog, Marketing agents |
| 5 | Orchestration | Multi-agent coordination |
| 6 | Knowledge Graphs | Product relationships |
| 7-8 | Production | Full system, Gradio UI |

### Production-Ready Features

- **Full Observability**: Every API call traced to Langfuse
- **Structured Outputs**: Pydantic models for type safety
- **Parallel Processing**: Concurrent agent execution
- **Error Handling**: Graceful failure with logging
- **User Interface**: Gradio for business users

### Next Steps for Production

1. Connect to real product APIs
2. Schedule regular competitive scans
3. Add alerting for price changes
4. Integrate with BI dashboards
5. Expand to more product categories